<a href="https://colab.research.google.com/github/miaflynn/CYPLAN255-Final-Project/blob/main/processing_notebooks/02e_processing_naics_nbhd_scatterplot_demographics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
from shapely.geometry import LineString

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!wget https://raw.githubusercontent.com/miaflynn/CYPLAN255-Final-Project/main/src/functions.py
import functions as fx

import importlib
importlib.reload(fx)

In [ ]:
gdf = gpd.read_file("/content/drive/MyDrive/C255_final_project/cleaned/open_close_after_2016.geojson")

In [ ]:
gdf.info()

## Neighborhood Level Analysis

In [ ]:
# Adding SF neighborhood geometry

nbhd_gdf = gpd.read_file(
    "/content/drive/MyDrive/C255_final_project/raw/sf_neighborhoods.geojson"
)

In [ ]:
nbhd_gdf.columns

In [ ]:


gdf = gdf.set_crs(epsg=4326)
nbhd_gdf = nbhd_gdf.to_crs(epsg=4326)



In [ ]:
gdf = fx.group_points_by_poly_naics_year(
    gdf,
    nbhd_gdf,
    id_col='nhood'
)

In [ ]:
gdf

In [ ]:
gdf.info()

# Baseline by number of businesses in each neighborhood

In [ ]:
# closures during COVID by neighborhood and naics_group
closures = (
    gdf[gdf['year'].between(2020, 2021)]
    .groupby(['nhood', 'naics_group'])['closed']
    .sum()
)

# openings during recovery by neighborhood and naics_group
openings = (
    gdf[gdf['year'].between(2022, 2025)]
    .groupby(['nhood', 'naics_group'])['opened']
    .sum()
)

# biz_stock as baseline
biz_stock = (
    gdf
    .groupby(['nhood', 'naics_group'])['biz_stock']
    .first()
)

rate_table = pd.DataFrame({
    'biz_stock': biz_stock,
    'closures_2020_21': closures,
    'openings_2022_25': openings
}).fillna(0)

rate_table['closure_rate'] = rate_table['closures_2020_21'] / rate_table['biz_stock']
rate_table['recovery_rate'] = rate_table['openings_2022_25'] / rate_table['biz_stock']

rate_table

## Calculate Business Resilience Rate

In [ ]:
# Filtering to nbhds with at least 50 businesses
rate_table_filtered = rate_table[rate_table['biz_stock'] >= 50]

# Demographic Data by Neighborhood

In [ ]:
import geopandas as gpd
from shapely import wkt

# Read in tract to neighborhood geometry file
neighborhoods = pd.read_csv('/content/drive/MyDrive/C255_final_project/raw/tract_to_neighborhood_geom.csv')

# Convert WKT geometry column to shapely geometry and make GeoDataFrame
neighborhoods["geometry"] = neighborhoods["the_geom"].apply(wkt.loads)
neighborhoods = gpd.GeoDataFrame(neighborhoods, geometry="geometry")

In [ ]:
neighborhoods = neighborhoods[[
    "geoid",
    "neighborhoods_analysis_boundaries",
    "geometry"
]].rename(columns={
    "geoid": "tract",
    "neighborhoods_analysis_boundaries": "neighborhood"
})

In [ ]:
neighborhoods.head()

In [ ]:
!pip install census
from census import Census
import pandas as pd

# Prepare Census Bureau API for data pulling
api_key = 'c9ad2fda3c683e4a14992e89daed0afee55738d2'
c = Census(key=api_key)

In [ ]:
race_vars = {
    "B03002_001E": "population",
    "B03002_003E": "white",           # Non-Hispanic White
    "B03002_004E": "black",           # Non-Hispanic Black
    "B03002_005E": "aian",            # Non-Hispanic American Indian/Alaska Native
    "B03002_006E": "asian",           # Non-Hispanic Asian
    "B03002_007E": "nhpi",            # Non-Hispanic Native Hawaiian/Pacific Islander
    "B03002_008E": "other",           # Non-Hispanic Some Other Race
    "B03002_012E": "latina_o",        # Hispanic or Latino
}

income_vars = {
    "B19013_001E": "median_income"
}

all_vars = {**race_vars, **income_vars}

# Pulling ACS 5-year 2023 data
acs = pd.DataFrame(
    c.acs5.get(
        list(all_vars.keys()),
        {'for': 'tract:*', 'in': 'state:06 county:075'},
        year=2023
    )
).rename(columns=all_vars)

acs['tract'] = '06075' + acs['tract']

In [ ]:
# Replace Census null codes with NaN
acs['median_income'] = acs['median_income'].replace(-666666666, pd.NA)

In [ ]:
acs.head()

In [ ]:
neighborhoods.head()

In [ ]:
acs['tract'] = acs['tract'].astype(str).str.lstrip('0')

neighborhoods['tract'] = neighborhoods['tract'].astype(str)
acs['tract'] = acs['tract'].astype(str)

merged = neighborhoods.merge(acs, on='tract', how='inner')

# Merge with neighborhoods shapefile
merged = neighborhoods.merge(acs, on='tract', how='inner')

# Aggregate by neighborhood
agg_cols = {
    'population': 'sum',
    'white': 'sum',
    'black': 'sum',
    'aian': 'sum',
    'asian': 'sum',
    'nhpi': 'sum',
    'other': 'sum',
    'latina_o': 'sum',
    'median_income': 'mean'
}

final = merged.groupby('neighborhood').agg(agg_cols).reset_index()



In [ ]:
# Percentages
# combining Native American into other because all the values are less than 1%
# combining NHPI with Asian because all the values are less than .02%
final['other'] = final['other'] + final['aian']
final = final.drop(columns='aian')
final['asian'] = final['asian'] + final['nhpi']
final = final.drop(columns='nhpi')

final['pct_other']   = final['other']   / final['population']
final['pct_white']   = final['white']   / final['population']
final['pct_black']   = final['black']   / final['population']
final['pct_asian']   = final['asian']   / final['population']
final['pct_latina_o']= final['latina_o']/ final['population']

# Filter out very small nbhds
final_df = final[final['population'] >= 200]

final_df

In [ ]:
df = rate_table_filtered.copy()
df['resilience'] = df['recovery_rate'] - df['closure_rate']

final_indexed = final.set_index('neighborhood')
final_indexed['pct_poc'] = 1 - final_indexed['pct_white']
demo_cols = ['median_income', 'pct_poc']

df = df.join(final_indexed[demo_cols], on='nhood', how='left')
df['median_income'] = pd.to_numeric(df['median_income'], errors='coerce')

df = df.sort_values('resilience', ascending=False)


In [ ]:
df.head()

In [ ]:
pct_cols = ['neighborhood', 'population', 'median_income',
            'pct_white', 'pct_black', 'pct_asian', 'pct_latina_o', 'pct_other']
final_slim = final_df[pct_cols].set_index('neighborhood')

df_reset = df.reset_index()
df_reset = df_reset.rename(columns={'nhood': 'neighborhood'})

combined = df_reset.merge(final_slim, on='neighborhood', how='inner')

combined = combined.drop(columns='median_income_y').rename(columns={'median_income_x': 'median_income'})

combined.to_parquet('sf_business_demographics_nhood_naics.parquet')

print(combined.shape)
combined.head()

In [ ]:
gdf.to_parquet('sf_businesses_nhood_naics.parquet')